# 17 - Pass 8c: a diffusion model in GAN-BERT's place (one commit, GPU)

Same structure as notebooks 08 and 11-13: code, tests, data check, two runs at a time, then **the analysis
runs inside this notebook** so the result can be judged straight away.

**What Pass 8c asks** (vault: Experiment Plan, "Pass 8 plan"): generative AI is more than LLMs. GAN-BERT's
generator makes fake [CLS] vectors in one shot and is trained to fool a discriminator; it never helped here
(Passes 1-3, GAN check). A **diffusion model** is the generative family behind today's image generators: it
learns to remove noise from real vectors and makes new ones by denoising step by step. It is trained with a
plain regression loss, never adversarially. Here it is a small network over BERT's [CLS] vectors (not text,
so it cannot leave the AI writing style that sank the LLM pairs).

| Arm | What it is | Compared with |
|---|---|---|
| A24 | GAN-BERT's discriminator (k real classes + "fake"), with the fakes drawn from the diffusion model | A4 (GAN fakes, Pass 1), A1 |
| A25 | plain BERT on L, plus a **consistency** loss on U: each U vector is noised part-way and the diffusion model maps it back to a nearby realistic vector; the sarcasm head must give both the same prediction | A1 |
| A26 | control for A25: the same noise level, but no denoising (plain Gaussian noise) | A25 - A26 = what the generative model adds |
| A1, 10%, seed 0, tag `rerun` | plain BERT | must equal Pass 1's A1_p10_s0 exactly (the training loop changed) |

5 seeds, 1% and 10% labels, test and masked tests scored. **Nothing is picked on test**: every diffusion
setting (100 noise levels, 10 sampling steps, round trip from level 30 in 3 steps, consistency weight 1.0,
denoiser width 1024, lr 1e-3) is fixed in `src/gainsplit/diffusion.py` and `plans.py` before running.

**Before you run, in the right-hand panel:**
- **Input:** the same `gainsplit-processed` version as notebooks 10-14.
- **Internet: on**, **Accelerator: GPU T4 x2**.
- Then **Save Version -> Save & Run All (Commit)**. About 30-40 minutes; it does not depend on notebooks 15-16.

## 1. Get the code
Clones the public repo and prints the **commit**. Every run records it in its `result.json`.

In [ ]:
import os, subprocess, sys, time
T_START = time.time()
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

## 2. Tests (about two minutes)
The repo's tests train a tiny random BERT for a few steps on made-up data. For Pass 8 they also run a
tiny MLM and JEPA pretraining, check that pretraining never reads a label, train the new arms from the
saved encoder, and run the three diffusion arms. They prove the pieces fit together before real GPU
time is spent. If anything fails, stop and paste the output.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed: see output above"

## 3. Check the data on the CPU (pre-flight)
Checks every file against `manifest.json` and `manifest_pass3.json`. Any problem raises an error here.

In [ ]:
import json
from pathlib import Path
import torch
from gainsplit.data import find_data_dir, verify, preflight
from gainsplit.plans import PLANS

PLAN = "pass8_diff"
OUT_DIR = f"/kaggle/working/results/{PLAN}"
configs = PLANS[PLAN]()

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
DATA_DIR = find_data_dir("/kaggle/input")
verify(DATA_DIR)
print("data:", DATA_DIR, "- all files match manifest.json")
display(preflight(configs, DATA_DIR))

## 4. The list of runs and the time estimate
31 runs: A24, A25, A26 at 10% (about 2-3 min each, like the GAN) then at 1% (about 1 min each), then the A1
re-run. About 60 run-minutes, so **about 30-40 minutes on two GPUs**.

In [ ]:
from gainsplit.diffusion import SETTINGS
print("diffusion settings:", SETTINGS)
for c in configs:
    print(f"{c.run_id:18s} arm={c.arm:3s} labels={c.budget:3d}% seed={c.seed} cons_weight={c.cons_weight:g}")
print(len(configs), "runs")

## 5. Run
One run per GPU; when a GPU frees up, the next run starts. If the session stops halfway, run this cell again:
finished runs are skipped.

In [ ]:
from gainsplit.runner import run_all
table = run_all(configs, DATA_DIR, OUT_DIR)
table.to_csv(f"{OUT_DIR}/summary.csv", index=False)
print(f"total so far: {(time.time() - T_START) / 60:.1f} min")

## 6. Checks and analysis (CPU, about 3 minutes)
1. **Re-run check:** A1_p10_s0_rerun must equal Pass 1's A1_p10_s0 exactly (proves the changed training loop
   leaves the old arms alone).
2. **Diffusion health** (last training-curve point, mean over seeds): A24 `p_fake_fake` (how sure the
   discriminator is that diffusion samples are fake; near 1 = easy fakes), `fake_mean_gap_sd` (distance of the
   fakes' mean from the real mean, in sd); A25/A26 `aug_shift_sd` (how far the augmented vector moves),
   `flip_rate` (how often the augmented vector changes the predicted class), `loss_diff` (denoising loss).
3. **Means per arm** and the **paired bootstrap** (10,000 resamples, Holm within this table), contrasts fixed
   in `src/gainsplit/analysis_pass8.py` before running; `seed_wins` per contrast.

In [ ]:
from gainsplit import analysis_pass8 as a8
import pandas as pd
pd.set_option("display.width", 200)
RESULTS_DIRS = [f"{REPO_DIR}/results/pass1", f"{REPO_DIR}/results/pass2b", OUT_DIR]
ANALYSIS_DIR = f"{OUT_DIR}/analysis"
tables = a8.run_all(RESULTS_DIRS, "diff", ANALYSIS_DIR)
for name, t in tables.items():
    print(f"\n=== {name}")
    display(t)
rr = tables.get("rerun_check")
print("re-run check:", "PASS" if rr is not None and len(rr) and rr["ok"].all() else "DIFFERENCE or missing - report this first")

## 7. Verdict against the predictions written before running
Supported only if the difference is positive **and** significant after Holm (p_holm < 0.05). Paste this
cell's output when you report.

In [ ]:
HYPOTHESES = [('H8c.1', 'A24-A4 (diffusion vs GAN fakes)', 1, 'diffusion fakes vs GAN fakes (no direction predicted)'), ('H8c.1', 'A24-A4 (diffusion vs GAN fakes)', 10, 'diffusion fakes vs GAN fakes (no direction predicted)'), ('H8c.2', 'A24-A1', 1, 'the k+1 discriminator with diffusion fakes beats plain BERT (predicted: no)'), ('H8c.2', 'A24-A1', 10, 'the k+1 discriminator with diffusion fakes beats plain BERT (predicted: no)'), ('H8c.3', 'A25-A1', 1, 'diffusion consistency on U helps at 1%'), ('H8c.3', 'A25-A1', 10, '... and at 10%'), ('H8c.4', 'A25-A26 (denoising vs plain noise)', 1, 'the generative model beats plain noise at 1%'), ('H8c.4', 'A25-A26 (denoising vs plain noise)', 10, '... and at 10%')]
f1 = tables["contrasts_f1"].set_index(["contrast", "budget"])
wins = tables["seed_wins"].set_index(["contrast", "budget"])
for h, name, b, text in HYPOTHESES:
    if (name, b) not in f1.index:
        print(f"{h} {name} at {b}%: no runs (skipped)"); continue
    r, w = f1.loc[(name, b)], wins.loc[(name, b)]
    if "no direction" in text:
        verdict = "significant difference" if r.p_holm < 0.05 else "no significant difference"
    else:
        verdict = ("SUPPORTED" if r.mean_diff > 0 and r.p_holm < 0.05 else
                   "opposite, significant" if r.mean_diff < 0 and r.p_holm < 0.05 else "not supported (no significant difference)")
    print(f"{h} {name:34s} {b:3d}%  {r.mean_diff:+.2f} points [{r.ci_low:+.2f}, {r.ci_high:+.2f}]  "
          f"p_holm {r.p_holm:.3f}  seeds ahead {w.seeds_ahead}/{w.n_seeds}  -> {verdict}   ({text})")

## Download
Zips the results folder (runs, logs, pretraining receipt, analysis tables). A commit keeps it in the
Output tab: download `results_<plan>.zip` and send it with the notebook log.

In [ ]:
import shutil
print(shutil.make_archive(f"/kaggle/working/results_{PLAN}", "zip", OUT_DIR))
print(f"notebook time: {(time.time() - T_START) / 60:.1f} min")